In [1]:
%cd ..

/Users/macos/Uni/1st_year/IntroDS/mini_project


In [2]:
import re
from itertools import product
from typing import Union

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from tqdm.notebook import tqdm
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.arima.model import ARIMA

In [3]:
plt.style.use('seaborn-v0_8')
plt.rcParams.update({'font.size': 8})

In [4]:
import warnings
warnings.filterwarnings("ignore")

# Load data

In [5]:
path_data = "res/data_merged_v1.2.csv"
path_countries = "res/countries.txt"

## Load country list

In [6]:
with open(path_countries, 'r') as file:
    countries = file.read()
country_list = countries.split('\n')
country_list = [country.strip() for country in country_list]

### Load main data

In [7]:
df = pd.read_csv(path_data)
df.head()

,Entity,Year,gdp_growth,gdp_per_capita,Electricity from renewables (TWh),Capacity(MW),Investment(USDmn)
0,Algeria,2011,2.9,5455.679403,0.50,252.6,0.00
1,Algeria,2012,3.4,5592.220115,0.62,252.6,0.94
2,Algeria,2013,2.8,5499.587331,0.33,252.6,0.94
3,Algeria,2014,3.8,5493.056695,0.25,254.9,0.93
4,Algeria,2015,3.7,4177.889542,0.22,302.9,0.89


# ARIMA model

In [8]:
factor_RP = 'Electricity from renewables (TWh)' # Column label used for predicting resource potential
factor_TP = 'Capacity(MW)'                      # Column label used for predicting Technical Potential
factor_MP = 'Investment(USDmn)'                 # Column label used for predicting Market Potential

In [9]:
# Define searching list of parameters
ps = range(1, 5, 1)
qs = range(1, 5, 1)
order_list = list(product(ps, qs))

def optimize_ARMA(endog: Union[pd.Series, list]) -> pd.DataFrame:
    results = []
    for order in order_list:
        try:
            model = SARIMAX(endog, order=(order[0], 0, order[1]), simple_differencing=False).fit(disp=False)
  
            aic = model.aic
            results.append([order, aic])

            result_df = pd.DataFrame(results)
            result_df.columns = ['(p,q)', 'AIC']

            # Sort in ascending order, lower AIC is better
            result_df = result_df.sort_values(by='AIC', ascending=True).reset_index(drop=True)

        except:
            continue

    return result_df


In [110]:
EPS = 1e-6

def forecast(
    df: pd.DataFrame,
    country: str,
    trg_name: str,
    to_year: int = 2030,
) -> pd.Series:
    """Forecast the potential

    Args:
        df (pd.DataFrame): data
        country (str): coutry to calculate potential
        trg_name (str): factor name
        to_year (int, optional): Year to which the forecast is run. Defaults to 2030.

    Returns:
        pd.Series: forecasted data
    """

    df_country = df[df['Entity'] == country]
    s = df_country[[trg_name, 'Year']].set_index('Year')

    # Apply differencing to remove unstationary factor
    year_start, year_end = s.index.min(), s.index.max()
    s_diff = pd.Series(s[trg_name].diff(), index=range(year_start + 1, year_end + 1))

    # Make forecast
    result = optimize_ARMA(s_diff)
    for p,q in result['(p,q)']:
        try:
            arima1 = ARIMA(s_diff, order=(p, 1, q))
            model = arima1.fit()

            break
        except Exception:
            continue
    forecast_period = to_year - year_end.item()
    pred = model.forecast(forecast_period)

    # Undiff
    s_final_diff = pd.concat((s_diff, pred))
    s_final_undiff = [s.iloc[0].item()]
    for i in range(1, len(s_final_diff) + 1):
        s_final_undiff.append(s_final_undiff[i - 1] + s_final_diff.iloc[i - 1])
    s_final_undiff = pd.Series(s_final_undiff, index=range(year_start, to_year + 1))

    # Calculate potential
    potential = pd.Series(
        np.clip(
            s_final_undiff.values[1:] / (s_final_undiff.values[:-1] + EPS),
            a_min=-20,
            a_max=20
        ),
        index=range(year_start+1, to_year + 1)
    )
    
    return potential

In [116]:
df_result = pd.DataFrame(columns=['country', 'Year','RP','MP','TP'])

with tqdm(country_list) as pbar:
    for country in country_list:
        pbar.set_description(country)

        if len(df[df['Entity'] == country]) == 0:
            pbar.update()
            continue

        rp = forecast(df,country,factor_RP)
        mp = forecast(df, country, factor_MP)
        tp = forecast(df,country,factor_TP)

        df_tmp = pd.DataFrame({
            'country': country,
            'Year': rp.index,
            'RP': rp.values,
            'MP': mp.values,
            'TP': tp.values
        })

        df_result = pd.concat([df_result, df_tmp])

        pbar.update()

        # if country == 'Algeria':
        #     break
        
df_result['REI'] = (df_result['RP'] + df_result['MP'] + df_result['TP']) / 3

  0%|          | 0/60 [00:00<?, ?it/s]

### Mark which entry is past/forecasted

In [117]:
idx_past = df_result[df_result['Year'] <= 2020].index
idx_future = df_result[df_result['Year'] > 2020].index

df_result.loc[idx_past, 'Prediction'] = 'Past'
df_result.loc[idx_future, 'Prediction'] = 'Forecast'

In [121]:
len(df_result)

1064

In [123]:
df_result.to_csv("res/Oct17_processed_v1.csv", index=False)